In [2]:
import os

# Change this to your actual project root path
project_root = "E:/HealthyBites"
os.chdir(project_root)

print(f"Now working in: {os.getcwd()}")

Now working in: E:\HealthyBites


In [3]:
import pandas as pd
import ast

# File path to your master split dataset
file_path = "data/processed/healthybites_master_dataset_split.csv"

print("Loading dataset...")
df = pd.read_csv(file_path)

# Helper function to ensure we are working with real Python lists
def force_list(x):
    if isinstance(x, list):
        return x
    try:
        return ast.literal_eval(x)
    except:
        return []

df['core_ingredients'] = df['core_ingredients'].apply(force_list)
df['pantry_ingredients'] = df['pantry_ingredients'].apply(force_list)

print(f"Successfully loaded {len(df)} recipes.")

Loading dataset...
Successfully loaded 200000 recipes.


In [ ]:
def fix_ingredient_classification(row):
    core = row['core_ingredients']
    pantry = row['pantry_ingredients']
    
    # List of ingredients that SHOULD be in Core but are stuck in Pantry
    targets_to_move = ["ground beef", "beef", "hamburger", "sirloin", "chicken broth"]
    
    for item in targets_to_move:
        # Check if the item exists in the pantry list
        if item in pantry:
            pantry.remove(item)
            # Add to core if not already there
            if item not in core:
                core.append(item)
                
    return core, pantry

print("Correcting ingredient classifications...")
# Apply the fix to the dataframe
corrections = df.apply(fix_ingredient_classification, axis=1)
df['core_ingredients'] = [x[0] for x in corrections]
df['pantry_ingredients'] = [x[1] for x in corrections]

print("Correction complete.")

Correcting ingredient classifications...
Correction complete.


In [10]:
# Search for recipes that contain ground beef in the core ingredients
test_search = df[df['core_ingredients'].apply(lambda x: "ground beef" in x)]

if not test_search.empty:
    print("Verification Successful! Sample recipes with corrected core:")

else:
    print("Ground beef not found in core. Check your spelling or target list.")

Verification Successful! Sample recipes with corrected core:


In [ ]:
test_search = df[df['core_ingredients'].apply(lambda x: "chicken broth" in x)]

if not test_search.empty:
    print("Verification Successful! Sample recipes with corrected core:")

else:
    print("Ground beef not found in core. Check your spelling or target list.")

In [9]:
# Convert lists back to strings so they save correctly in CSV format
df['core_ingredients'] = df['core_ingredients'].apply(str)
df['pantry_ingredients'] = df['pantry_ingredients'].apply(str)

df.to_csv(file_path, index=False)
print(f"Changes saved to {file_path}. You can now restart your Retrieval Engine!")

Changes saved to data/processed/healthybites_master_dataset_split.csv. You can now restart your Retrieval Engine!


In [ ]:
import pandas as pd

# Load the split file and the raw file
split_file = "data/processed/healthybites_master_dataset_split.csv"
df_split = pd.read_csv(split_file)

# CHECK: If core_ingredients is missing, we need to reload it from your backup
# or re-run your split_core_pantry logic. 
if 'core_ingredients' not in df_split.columns:
    print("Column missing! Attempting to restore...")
    # If you have a backup of the split file before the 'minutes' merge, load it here.
    # Otherwise, you must re-run your notebook that creates core_ingredients.
else:
    print("Columns found. Ensuring 'minutes' is present...")

# Re-save correctly (Ensuring all columns are preserved)
# If minutes is already there, this just ensures the file is healthy.
df_split.to_csv(split_file, index=False)
print("File check complete. Available columns:", df_split.columns.tolist())